# UNIVERSITÉ DE DJIBOUTI
**Master 2 Intelligence Artificielle et Modélisation des Données – IAMD**

# Traitement Automatique du Langage Naturel
## Projet de fin de module – Binôme 5
### Reconnaissance d'entités nommées propres à Djibouti
*spaCy NER + EntityRuler : un enrichissement local améliore-t-il un NER générique ?*

| | |
|---|---|
| **Binôme** | Nom Prénom 1 – Nom Prénom 2 |
| **Enseignant** | Dr Houssein Ahmed Assowe – Directeur du CRMN |
| **Dépôt final** | lundi 23 novembre 2026, 18 h |
| **Soutenance** | mardi 24 novembre 2026 |

## 1 Objectif général

> **Question de recherche.** Un modèle NER français généraliste (spaCy `fr_core_news_sm`) reconnaît-il
> les institutions, lieux, formations, dates et personnes propres à Djibouti ? Un enrichissement local
> (règles + lexique) améliore-t-il les résultats, et où le système se trompe-t-il encore ?

**Hypothèse (à rédiger AVANT l'Étape 6, comme au TP5/TP8).**
H1 : *L'ajout de règles génériques et d'un lexique local améliore la reconnaissance des entités propres à Djibouti
par rapport à un modèle NER français généraliste, surtout pour les institutions, les lieux et les formations ;
cette amélioration peut s'accompagner de faux positifs.*
H2 : *Le ré-entraînement du NER de spaCy sur un corpus annoté localement adapte mieux le modèle aux étiquettes locales
(dont DATE et FORM) qu'un simple ajout de règles.*
*(Propositions de départ : relisez-les et reformulez-les avec vos mots.)*

**Compétences travaillées :**
- constituer et documenter un corpus annoté (provenance, guide d'annotation, séparation dev/test) ;
- appliquer un NER pré-entraîné et un `EntityRuler` ;
- construire un lexique local **sans fuite de données** ;
- évaluer par précision, rappel et F1 (par étiquette et globalement) ;
- analyser au moins 10 erreurs et formuler des limites honnêtes ;
- produire une démonstration exécutable.

**Systèmes comparés**

| Système | Description | Rôle | Question à laquelle il répond |
|---|---|---|---|
| A | spaCy `fr_core_news_sm` seul | point de référence | Que sait reconnaître un modèle généraliste sans adaptation ? |
| B | spaCy + `EntityRuler` à règles **génériques** | **baseline obligatoire** | Des règles linguistiques génériques améliorent-elles la détection ? |
| C | B + lexique local construit sur le **train** | approche améliorée | Les connaissances propres à Djibouti apportent-elles un gain ? |
| D | NER de spaCy **ré-entraîné** sur le train (arrêt sur la validation) | approche apprise | L'apprentissage sur les données locales est-il plus efficace ? |

**Calendrier restant :** 22 nov démonstration prête · 23 nov 18 h dépôt · 24 nov soutenance.

## 2 Préparation de l'environnement

Dans un terminal, à la racine du projet :
```
pip install -r requirements.txt
python -m spacy download fr_core_news_sm
```

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

SEED = 42  # graine fixée pour toute expérience aléatoire

ROOT = Path.cwd()
if not (ROOT / "data" / "corpus").exists():  # notebook lancé depuis un sous-dossier
    ROOT = ROOT.parent
CORPUS = ROOT / "data" / "corpus"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
LABELS = ["ORG", "LOC", "PER", "DATE", "FORM"]

print("Racine du projet :", ROOT)

## 3 Étape 1 – Charger et diagnostiquer le corpus

Le corpus (137 phrases, 331 entités) réunit des textes publics en français : l'article « Djibouti » de
Wikipédia (CC BY-SA 4.0), des arrêtés du Journal officiel (masters, 2026), deux rapports Hcéres (Université de Djibouti
et licence LALT de l'IUT Tertiaire), la loi d'orientation de l'éducation et des exemples du cours
(voir `data/raw/SOURCES.md` et `data/corpus/DATA_CARD.md`).

Découpage **60 / 20 / 20 par blocs** (source + section, jamais phrase par phrase, graine 42) :
- **train** : construit le lexique de C et entraîne le modèle D ;
- **val** (validation) : règle les règles de B et C, et arrête l'entraînement de D ;
- **test** : n'est utilisé qu'une fois, à l'Étape 6.

In [ ]:
def charger(nom):
    """Reconstruit la liste de phrases {id, section, text, entities} depuis les deux CSV."""
    phrases = pd.read_csv(CORPUS / "sentences.csv", encoding="utf-8")
    entites = pd.read_csv(CORPUS / "entities.csv", encoding="utf-8")
    phrases = phrases[phrases["split"] == nom]
    entites = entites[entites["split"] == nom]
    par_id = {i: g for i, g in entites.groupby("id")}
    docs = []
    for r in phrases.itertuples():
        g = par_id.get(r.id)
        ents = [] if g is None else [
            {"start": int(e.start), "end": int(e.end), "label": e.label, "text": e.text}
            for e in g.itertuples()]
        docs.append({"id": r.id, "section": r.section, "text": r.text, "entities": ents})
    return docs


train, val, test = charger("train"), charger("val"), charger("test")


def table_entites(docs, split):
    return pd.DataFrame([
        {"split": split, "id": d["id"], "section": d["section"], **e}
        for d in docs for e in d["entities"]
    ])


ents = pd.concat([table_entites(train, "train"), table_entites(val, "val"), table_entites(test, "test")],
                 ignore_index=True)

print("Phrases :", {"train": len(train), "val": len(val), "test": len(test)})
print(pd.crosstab(ents["label"], ents["split"], margins=True)[["train", "val", "test", "All"]])

In [ ]:
# Répartition des entités par étiquette et par ensemble (visualisation du déséquilibre)
import matplotlib.pyplot as plt
rep_ent = pd.crosstab(ents["label"], ents["split"])[["train", "val", "test"]].loc[["ORG", "LOC", "DATE", "PER", "FORM"]]
ax = rep_ent.plot.bar(figsize=(7, 3.8), rot=0)
ax.set_ylabel("nombre d'entités"); ax.set_title(f"Répartition des {len(ents)} entités annotées")
FIGURES.mkdir(exist_ok=True)
plt.tight_layout(); plt.savefig(FIGURES / "repartition_entites.png", dpi=150); plt.show()

**Sortie attendue (corpus actuel)**
```
Phrases : {'train': 84, 'val': 26, 'test': 27}
split  train  val  test  All
label                       
DATE      55    6    18   79
FORM       4   10     5   19
LOC       83    8    13  104
ORG       51   27    28  106
PER       14    5     4   23
All      207   56    68  331
```

**Questions d'analyse**
1. Quelle étiquette est la moins représentée dans le train ? Que cela implique-t-il pour un lexique construit sur le train ?
2. Pourquoi le F1 de la classe PER sera-t-il peu interprétable sur le test (support = 4) ?
3. Pourquoi découper par **blocs** (source + section) plutôt qu'au hasard phrase par phrase ?
4. Pourquoi faut-il une **validation** en plus du test ?

### 3.1 – Convention d'annotation

| Étiquette | Ce qu'on annote | Exemple | Justification |
|---|---|---|---|
| ORG | institutions, administrations, organisations, partis, hôpitaux, établissements, facultés, forces armées, médias | Université de Djibouti | institution |
| LOC | pays, régions, villes, quartiers, reliefs, plans d'eau, infrastructures, territoires historiques | Lac Assal | site géographique |
| PER | personnes nommées, sans titre ni fonction | Ismaïl Omar Guelleh | personne |
| DATE | dates complètes, jour + mois, mois + année, années, périodes nommées | 24 novembre 2026 | expression temporelle |
| FORM | intitulés de diplômes ou parcours, sigles compris | Master IAMD | formation universitaire |

Règles : on annote l'expression **la plus longue**, **sans entités imbriquées** ; un nom générique sans nom propre n'est pas annoté ;
une plage « du 28 au 30 janvier 2019 » est **une seule** DATE ; « Djibouti » est LOC (pays ou ville).
**Cas ambigus tranchés :** institution portant un nom de lieu (« Université de Djibouti ») = **ORG**, le lieu à l'intérieur n'est pas annoté à part ;
« port de Djibouti » = LOC (infrastructure) ; un titre ou une fonction n'est pas dans l'entité PER ; un sigle de diplôme (LALT) = FORM.
Voir aussi `data/corpus/DATA_CARD.md`.

## 4 Étape 2 – Garde-fou anti-fuite : que la validation et le test ne connaissent-ils pas du train ?

On mesure la part des entités de val et de test déjà vues dans le train. Cette part conditionne l'interprétation :
un bon score sur des entités *déjà vues* mesure la mémorisation du lexique, pas la généralisation.

In [ ]:
vus_train = set(ents.query("split == 'train'")[["text", "label"]].itertuples(index=False, name=None))
for nom in ("val", "test"):
    t = ents.query(f"split == '{nom}'").copy()
    t["vue_en_train"] = [(x, l) in vus_train for x, l in zip(t["text"], t["label"])]
    print(f"--- {nom} ---")
    print(t.groupby("label")["vue_en_train"].agg(vues="sum", total="count"))
    print("Entités distinctes jamais vues dans le train :",
          t.loc[~t["vue_en_train"], ["text", "label"]].drop_duplicates().shape[0], "\n")

**Consigne importante.** À partir d'ici : le **train** construit le lexique et entraîne D ; la **validation** sert à
régler (règles, arrêt de l'entraînement) ; le **test** n'est regardé qu'à l'Étape 6, une seule fois.
Ne lisez jamais le test pour écrire une règle ou ajouter un mot au lexique. Toute règle doit pouvoir être justifiée
par le train ou par une connaissance générale (ex. « Université de … »).

## 5 Étape 3 – Système A : spaCy seul (référence)

Les scores de développement sont calculés sur la **validation** (le test attend l'Étape 6).

On charge `fr_core_news_sm` et on regarde **quelles étiquettes il sait produire**. Les étiquettes de
spaCy ne sont pas les nôtres : il faut une table de correspondance explicite.

In [ ]:
# Outils communs aux systèmes A, B, C et D (prédiction + évaluation à correspondance exacte)
REFS_TEST = {(d["id"], e["start"], e["end"], e["label"]) for d in test for e in d["entities"]}
REFS_VAL = {(d["id"], e["start"], e["end"], e["label"]) for d in val for e in d["entities"]}

# Étiquettes spaCy -> notre schéma. MISC est écartée (voir questions d'analyse) ;
# DATE et FORM n'existent que si une règle (EntityRuler) les produit.
MAPPING = {"PER": "PER", "LOC": "LOC", "ORG": "ORG", "MISC": None, "DATE": "DATE", "FORM": "FORM"}


def predire(nlp_modele, docs, mapping=MAPPING):
    """Retourne l'ensemble des (id_phrase, start, end, label) prédits, étiquettes déjà converties."""
    preds = set()
    for d, doc in zip(docs, nlp_modele.pipe([d["text"] for d in docs])):
        for ent in doc.ents:
            lab = mapping.get(ent.label_)
            if lab:
                preds.add((d["id"], ent.start_char, ent.end_char, lab))
    return preds


def evaluer(references, predictions, labels=LABELS):
    """references / predictions : ensembles de tuples (id_phrase, start, end, label).
    Correspondance exacte. Retourne un DataFrame (precision, rappel, f1, support) par étiquette + « micro »."""
    def prf(vp, fp, fn):
        p = vp / (vp + fp) if vp + fp else 0.0
        r = vp / (vp + fn) if vp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        return p, r, f
    lignes, tvp, tfp, tfn = [], 0, 0, 0
    for lab in labels:
        ref = {x for x in references if x[3] == lab}
        pre = {x for x in predictions if x[3] == lab}
        vp, fp, fn = len(ref & pre), len(pre - ref), len(ref - pre)
        tvp, tfp, tfn = tvp + vp, tfp + fp, tfn + fn
        p, r, f = prf(vp, fp, fn)
        lignes.append({"etiquette": lab, "precision": p, "rappel": r, "f1": f, "support": len(ref), "VP": vp, "FP": fp, "FN": fn})
    p, r, f = prf(tvp, tfp, tfn)
    lignes.append({"etiquette": "micro", "precision": p, "rappel": r, "f1": f, "support": tvp + tfn, "VP": tvp, "FP": tfp, "FN": tfn})
    # macro : moyenne simple des scores par étiquette (chaque classe pèse autant, quel que soit son effectif)
    par_lab = pd.DataFrame(lignes)
    lignes.append({"etiquette": "macro", "precision": par_lab["precision"].mean(), "rappel": par_lab["rappel"].mean(),
                   "f1": par_lab["f1"].mean(), "support": par_lab["support"].sum(),
                   "VP": tvp, "FP": tfp, "FN": tfn})
    return pd.DataFrame(lignes).set_index("etiquette").round(3)

In [ ]:
import spacy

try:
    nlp = spacy.load("fr_core_news_sm")
    print("Étiquettes du composant NER :", nlp.get_pipe("ner").labels)
except OSError:
    nlp = None
    print("Modèle absent -> python -m spacy download fr_core_news_sm")

if nlp is not None:
    pv_A = predire(nlp, val)
    print("Entités prédites par A :", len(pv_A), "| entités de référence (val) :", len(REFS_VAL))
    jamais = [l for l in LABELS if l not in {MAPPING[x] for x in nlp.get_pipe("ner").labels if MAPPING.get(x)}]
    print("Étiquettes de notre schéma que spaCy ne peut jamais produire :", jamais)
    display(evaluer(REFS_VAL, pv_A))

**Sortie attendue.** *(à remplir : liste des étiquettes spaCy, remarques)*

**Questions d'analyse**
1. Quelles étiquettes de notre schéma le modèle ne produit-il pas du tout ?
2. Que fait-on des entités `MISC` dans la comparaison ? Justifiez.

## 6 Étape 4 – Système B : baseline spaCy + EntityRuler (règles génériques)

**Baseline obligatoire.** On ajoute un `EntityRuler` **avant** le NER statistique (cf. TP4, § 8.2) avec
des règles qui ne dépendent d'aucun nom propre local : motifs de dates (regex/patterns), amorces
« Université de … », « Ministère de … », « lac … », etc.

In [ ]:
# Règles GÉNÉRIQUES : aucun nom propre du corpus. Seulement des gabarits (dates, mots-têtes d'institutions
# et de lieux, noms de diplômes). Justification : connaissance générale de la langue et de l'enseignement supérieur.
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre"]
ANNEE = r"^(18|19|20)\d{2}$"
JOUR = r"^(0?[1-9]|[12]\d|3[01]|1er)$"
PREPS = [[], [{"LOWER": p} for p in ("de",)], [{"LOWER": "du"}], [{"LOWER": "des"}],
         [{"LOWER": "d’"}], [{"LOWER": "d'"}], [{"LOWER": "de"}, {"LOWER": "la"}],
         [{"LOWER": "de"}, {"LOWER": "l’"}], [{"LOWER": "de"}, {"LOWER": "l'"}]]
TITRE = {"IS_TITLE": True}


def motifs_tete(tetes, label, sans_prep=True, tete_capitalisee=False, nmax=4):
    """tête + (préposition) + 1 à nmax mots capitalisés  ->  ex. « Université de Xxx », « lac Xxx »."""
    out = []
    for prep in PREPS:
        if not prep and not sans_prep:
            continue
        for n in range(1, nmax + 1):
            tete = {"LOWER": {"IN": tetes}}
            if tete_capitalisee:
                tete["IS_TITLE"] = True
            out.append({"label": label, "pattern": [tete] + prep + [dict(TITRE) for _ in range(n)]})
    return out


patterns_B = []
# DATE
patterns_B += [
    {"label": "DATE", "pattern": [{"TEXT": {"REGEX": JOUR}}, {"LOWER": {"IN": MOIS}}, {"TEXT": {"REGEX": ANNEE}}]},
    {"label": "DATE", "pattern": [{"TEXT": {"REGEX": JOUR}}, {"LOWER": {"IN": MOIS}}]},
    {"label": "DATE", "pattern": [{"LOWER": {"IN": MOIS}}, {"TEXT": {"REGEX": ANNEE}}]},
    {"label": "DATE", "pattern": [{"TEXT": {"REGEX": r"^(18|19|20)\d{2}[-–/](18|19|20)?\d{2}$"}}]},
    {"label": "DATE", "pattern": [{"LOWER": {"IN": ["début", "milieu", "fin"]}}, {"LOWER": {"IN": ["des", "du"]}},
                                  {"LOWER": "années"}, {"TEXT": {"REGEX": ANNEE}}]},
    {"label": "DATE", "pattern": [{"TEXT": {"REGEX": ANNEE}}]},
]
# ORG : mots-têtes d'institutions (tête capitalisée pour limiter les faux positifs)
TETES_ORG = ["université", "ministère", "institut", "faculté", "centre", "conseil", "agence", "banque", "hôpital",
             "lycée", "maternité", "clinique", "commission", "comité", "organisation", "autorité", "association", "ligue"]
patterns_B += motifs_tete(TETES_ORG, "ORG", sans_prep=False, tete_capitalisee=True)
# LOC : reliefs, plans d'eau, infrastructures
TETES_LOC = ["lac", "mont", "golfe", "désert", "plateau", "mer", "détroit", "océan", "île", "vallée", "baie", "port"]
patterns_B += motifs_tete(TETES_LOC, "LOC", sans_prep=True, tete_capitalisee=False, nmax=3)
# FORM : types de diplômes
patterns_B += [{"label": "FORM", "pattern": [{"TEXT": {"IN": ["DUT", "BTS", "DEUG", "Master", "Licence", "Doctorat"]}}]}]
for n in range(1, 5):
    patterns_B.append({"label": "FORM", "pattern": [{"LOWER": {"IN": ["master", "licence", "dut", "bts"]}}]
                       + [dict(TITRE) for _ in range(n)]})
    patterns_B.append({"label": "FORM", "pattern": [{"LOWER": {"IN": ["master", "licence"]}},
                       {"LOWER": {"IN": ["appliquée", "professionnelle", "mention"]}}] + [dict(TITRE) for _ in range(n)]})

print("Règles génériques de la baseline :", len(patterns_B), Counter(p["label"] for p in patterns_B))

if nlp is not None:
    nlp_b = spacy.load("fr_core_news_sm")
    ruler = nlp_b.add_pipe("entity_ruler", before="ner")   # AVANT le NER : les règles ont priorité
    ruler.add_patterns(patterns_B)
    print("Pipeline B :", nlp_b.pipe_names)
    pv_B = predire(nlp_b, val)
    display(evaluer(REFS_VAL, pv_B))

In [ ]:
def lister_erreurs(predictions, references=REFS_TEST, docs=test):
    """Faux positifs (FP) et faux négatifs (FN) d'un système, avec la phrase, pour l'analyse d'erreurs."""
    texte = {d["id"]: d["text"] for d in docs}
    lignes = []
    for kind, ens in (("FP", predictions - references), ("FN", references - predictions)):
        for (i, s, e, lab) in sorted(ens):
            lignes.append({"type": kind, "id": i, "label": lab, "span": texte[i][s:e], "phrase": texte[i]})
    return pd.DataFrame(lignes)


if nlp is not None:
    err_B = lister_erreurs(pv_B, REFS_VAL, val)
    pd.set_option("display.max_colwidth", 90, "display.width", 200)
    print(err_B["type"].value_counts().to_dict())
    display(err_B.drop(columns="phrase").sort_values(["type", "label", "id"]))

**Sortie attendue.** *(à remplir : nombre de règles, exemples d'entités gagnées)*

**Questions d'analyse**
1. Pourquoi l'ordre des composants (`before="ner"`) compte-t-il ?
2. Quelles règles génériques ont fait gagner le plus de rappel ? Lesquelles ont créé des faux positifs ?

## 7 Étape 5 – Système C : enrichissement local (lexique construit sur le train)

On extrait des entités du **train** pour construire un lexique de patterns (lieux, institutions,
personnes), puis on l'ajoute à la baseline.

**Consigne importante.** Le lexique est construit **uniquement** à partir de `train`. Sauvegardez-le dans
`results/lexique_local.csv` pour que l'expérience soit reproductible.

In [ ]:
# Lexique construit UNIQUEMENT sur le train. DATE est exclue : une date exacte du train ne généralise pas,
# les règles génériques de B s'en chargent déjà.
lexique = {lab: sorted({e["text"] for d in train for e in d["entities"] if e["label"] == lab})
           for lab in LABELS if lab != "DATE"}

# Garde-fou : une même forme ne doit pas avoir deux étiquettes dans le lexique
tous = [(t, lab) for lab, ts in lexique.items() for t in ts]
doublons = [t for t, n in Counter(t for t, _ in tous).items() if n > 1]
assert not doublons, f"formes à plusieurs étiquettes : {doublons}"
print({lab: len(ts) for lab, ts in lexique.items()}, "| total :", len(tous))

RESULTS.mkdir(exist_ok=True)
pd.DataFrame(tous, columns=["text", "label"]).sort_values(["label", "text"]) \
  .to_csv(RESULTS / "lexique_local.csv", index=False, encoding="utf-8")

patterns_C = patterns_B + [{"label": lab, "pattern": t} for t, lab in tous]   # B + lexique

if nlp is not None:
    nlp_c = spacy.load("fr_core_news_sm")
    ruler_c = nlp_c.add_pipe("entity_ruler", before="ner")
    ruler_c.add_patterns(patterns_C)
    pv_C = predire(nlp_c, val)
    display(evaluer(REFS_VAL, pv_C))      # score de développement (validation)

**Questions d'analyse**
1. Quel gain C apporte-t-il sur les entités *déjà vues* ? Et sur les entités *jamais vues* (Étape 2) ?
2. Quel risque y a-t-il à un lexique trop large (ambiguïté, faux positifs) ?

## 7 bis – Étape 5 bis – Système D : ré-entraîner le NER de spaCy (train / validation / test)

C'est ici que le découpage **train / validation / test** du cours prend tout son sens :
- on **entraîne** le composant NER de `fr_core_news_sm` sur le **train** (en lui ajoutant les étiquettes DATE et FORM) ;
- après chaque époque, on mesure le F1 sur la **validation** et on garde le meilleur modèle (arrêt anticipé) ;
- le **test** n'est lu qu'à l'Étape 6.

**Attention : oubli catastrophique.** Avec ~85 phrases, le modèle ré-entraîné peut perdre ce que A savait déjà.
Ce n'est pas un échec : c'est un résultat à discuter dans le rapport.

In [ ]:
import copy
import random
from spacy.training import Example

random.seed(SEED)
from spacy.util import fix_random_seed
fix_random_seed(SEED)   # graine spaCy/thinc : entraînement reproductible
MAX_EPOCHS, PATIENCE, BATCH, DROPOUT = 40, 6, 8, 0.3


def construire_exemples(nlp_modele, docs):
    """Convertit nos annotations (offsets de caractères) en Example spaCy. Retourne (exemples, nb_ignorées)."""
    exemples, ignorees = [], 0
    for d in docs:
        doc = nlp_modele.make_doc(d["text"])
        ents = []
        for e in d["entities"]:
            span = doc.char_span(e["start"], e["end"], label=e["label"], alignment_mode="strict")
            if span is None:        # la frontière ne coïncide pas avec un découpage en tokens
                ignorees += 1
            else:
                ents.append(span)
        doc.set_ents(ents)
        exemples.append(Example(nlp_modele.make_doc(d["text"]), doc))
    return exemples, ignorees


courbe = []   # (époque, perte, F1 micro validation)
if nlp is not None:
    nlp_d = spacy.load("fr_core_news_sm")
    ner_d = nlp_d.get_pipe("ner")
    for lab in ("DATE", "FORM"):
        ner_d.add_label(lab)
    ex_train, ignorees = construire_exemples(nlp_d, train)
    print(f"Exemples d'entraînement : {len(ex_train)} phrases ({ignorees} entités ignorées : frontière ≠ tokens)")

    meilleur_f1, meilleur_etat, sans_progres = -1.0, None, 0
    with nlp_d.select_pipes(enable=["tok2vec", "ner"]):
        optimiseur = nlp_d.resume_training()
        f0 = evaluer(REFS_VAL, predire(nlp_d, val)).loc["micro", "f1"]
        courbe.append((0, None, f0))
        print(f"époque  0 | F1 val = {f0:.3f}  (avant entraînement)")
        for epoque in range(1, MAX_EPOCHS + 1):
            random.shuffle(ex_train)
            pertes = {}
            for lot in spacy.util.minibatch(ex_train, size=BATCH):
                nlp_d.update(lot, sgd=optimiseur, drop=DROPOUT, losses=pertes)
            f1_val = evaluer(REFS_VAL, predire(nlp_d, val)).loc["micro", "f1"]
            courbe.append((epoque, pertes["ner"], f1_val))
            if f1_val > meilleur_f1:
                meilleur_f1, meilleur_etat, sans_progres = f1_val, nlp_d.to_bytes(), 0
            else:
                sans_progres += 1
            if epoque % 5 == 0 or sans_progres == 0:
                print(f"époque {epoque:2d} | perte = {pertes['ner']:8.1f} | F1 val = {f1_val:.3f}"
                      + ("  <- meilleur" if sans_progres == 0 else ""))
            if sans_progres >= PATIENCE:
                print(f"Arrêt anticipé à l'époque {epoque} (pas de progrès depuis {PATIENCE} époques)")
                break
        nlp_d.from_bytes(meilleur_etat)       # on garde le MEILLEUR modèle de la validation

    pv_D = predire(nlp_d, val)
    print(f"\nMeilleur F1 micro sur la validation : {meilleur_f1:.3f}")
    display(evaluer(REFS_VAL, pv_D))

In [ ]:
# Courbe d'apprentissage : F1 de validation (et perte) par époque
import matplotlib.pyplot as plt
FIGURES.mkdir(exist_ok=True)
if courbe:
    cdf = pd.DataFrame(courbe, columns=["epoque", "perte", "f1_val"])
    cdf.to_csv(RESULTS / "courbe_apprentissage.csv", index=False)
    fig, ax1 = plt.subplots(figsize=(7, 3.8))
    ax1.plot(cdf["epoque"], cdf["f1_val"], marker="o", color="tab:blue", label="F1 micro (validation)")
    ax1.set_xlabel("époque"); ax1.set_ylabel("F1 micro (validation)"); ax1.set_ylim(0, 1)
    ax2 = ax1.twinx()
    ax2.plot(cdf["epoque"], cdf["perte"], color="tab:red", alpha=.6, label="perte (train)")
    ax2.set_ylabel("perte NER (train)")
    ax1.set_title("Système D : courbe d'apprentissage")
    fig.tight_layout(); fig.savefig(FIGURES / "courbe_apprentissage.png", dpi=150); plt.show()

**Questions d'analyse**
1. À quelle époque le F1 de validation est-il maximal ? Que se passe-t-il ensuite (surapprentissage) ?
2. Pourquoi la perte d'entraînement continue-t-elle de baisser alors que le F1 de validation stagne ?
3. D bat-il A sur les étiquettes que A savait déjà produire (PER, LOC, ORG) ? Y a-t-il oubli catastrophique ?
4. Pourquoi la validation ne doit-elle pas être utilisée comme score final ?

## 8 Étape 6 – Évaluation : précision, rappel, F1

**C'est la seule étape où l'on lit le test.** Une entité prédite est un **vrai positif** si `(start, end, label)` est identique à une entité de référence
(*correspondance exacte*, cf. TP4 § 11). On calcule P, R et F1 **par étiquette** et en **micro-moyenne**.

In [ ]:
# Étape 6 : évaluation FINALE sur le TEST (une seule fois). evaluer(), predire() et REFS_TEST viennent plus haut.
resultats = {}
if nlp is not None:
    pred_A = predire(nlp, test)
    pred_B = predire(nlp_b, test)
    pred_C = predire(nlp_c, test)
    pred_D = predire(nlp_d, test)
for nom in ("A", "B", "C", "D"):
    pred = globals().get(f"pred_{nom}")
    if pred is not None:
        resultats[nom] = evaluer(REFS_TEST, pred)
        print(f"=== Système {nom} (test) ===")
        display(resultats[nom])

# Rappel sur entités déjà vues dans le train vs jamais vues (question 1 de l'Étape 5)
if resultats:
    tt = ents.query("split == 'test'").copy()
    tt["vue"] = [(x, l) in vus_train for x, l in zip(tt["text"], tt["label"])]

    def rappel_vu_jamais_vu(pred):
        tp = set(pred)
        tt["ok"] = [(i, s, e, l) in tp for i, s, e, l in zip(tt["id"], tt["start"], tt["end"], tt["label"])]
        return tt.groupby("vue")["ok"].agg(rappel="mean", n="count").rename(index={True: "déjà vues", False: "jamais vues"})

    comparaison = pd.concat({k: rappel_vu_jamais_vu(globals()[f"pred_{k}"]) for k in resultats}, axis=1).round(3)
    display(comparaison)

**Sortie attendue.** *(à remplir : un tableau P / R / F1 / support par étiquette, pour A, B, C et D)*

**Questions d'analyse**
1. Pourquoi ne faut-il pas conclure sur la classe PER (support = 4) ?
2. Que change une évaluation à correspondance *partielle* (chevauchement) par rapport à l'exacte ?
3. Pourquoi une seule valeur de F1 ne suffit-elle pas ?

## 9 Étape 7 – Tableau comparatif

Livrable obligatoire : un tableau d'au moins deux approches, dont la baseline. Sauvegarde dans
`results/tableau_comparatif.csv`.

In [ ]:
NOMS = {"A": "A – spaCy seul", "B": "B – + règles génériques", "C": "C – + lexique local (train)", "D": "D – spaCy ré-entraîné (train)"}
lignes = []
for k, r in resultats.items():
    ligne = {"système": NOMS[k], "P micro": r.loc["micro", "precision"], "R micro": r.loc["micro", "rappel"],
             "F1 micro": r.loc["micro", "f1"], "F1 macro": r.loc["macro", "f1"]}
    ligne.update({f"F1 {lab}": r.loc[lab, "f1"] for lab in LABELS})
    lignes.append(ligne)
tableau = pd.DataFrame(lignes)
RESULTS.mkdir(exist_ok=True)
tableau.to_csv(RESULTS / "tableau_comparatif.csv", index=False, encoding="utf-8-sig")
display(tableau)

In [ ]:
# Faux positifs / faux négatifs par étiquette et par système (test)
fpfn = pd.concat({k: r.loc[LABELS, ["FP", "FN"]] for k, r in resultats.items()}, axis=1)
fpfn.to_csv(RESULTS / "erreurs_par_etiquette.csv", encoding="utf-8-sig")
display(fpfn)

## 10 Étape 8 – Analyse qualitative d'au moins 10 erreurs

Livrable obligatoire. Sélectionnez au moins 10 cas variés (faux positifs, faux négatifs, mauvaises
frontières, mauvaise étiquette) et sauvegardez-les dans `results/erreurs_analysees.csv`.

| id | phrase | attendu | prédit | type d'erreur | cause probable | amélioration proposée |
|---|---|---|---|---|---|---|
| | | | | | | |

**Cas ambigus voulus du corpus :** « port de Djibouti » (LOC ou ORG ?), « La Nation » (ORG, titre de
presse), « forces armées djiboutiennes », « Djibouti » ville vs pays.

**Choix du système analysé.** Il est sélectionné sur la **validation** (jamais sur le test). Les erreurs sont classées en quatre familles :
faux négatif (entité non détectée), faux positif (mot ordinaire pris pour une entité), erreur de frontière (entité partiellement détectée)
et erreur de classification (bonne frontière, mauvaise étiquette). La colonne `impact` dit si l'erreur touche la précision, le rappel ou les deux ;
`en_face` montre ce que le système a prédit à la place (ou ce que la référence contenait).


In [ ]:
# 1) Système à analyser : choisi sur la VALIDATION (pas sur le test, pour ne pas sélectionner a posteriori).
#    Remarque : le F1 de validation de D est un peu optimiste, car la validation a servi à arrêter son entraînement.
f1_val = {k: evaluer(REFS_VAL, globals()[f"pv_{k}"]).loc["micro", "f1"] for k in "ABCD"}
meilleur_nom = max(f1_val, key=f1_val.get)
print("F1 micro de validation :", f1_val, "-> système analysé :", meilleur_nom)
meilleur = globals()[f"pred_{meilleur_nom}"]   # ses prédictions sur le TEST : on n'y regarde que les erreurs finales


def chevauche(a, b):
    return a[0] == b[0] and a[1] < b[2] and b[1] < a[2]


def famille_erreurs(predictions, references):
    """4 familles : faux négatif, faux positif, erreur de frontière, erreur de classification."""
    texte = {d["id"]: d["text"] for d in test}
    lignes = []
    for kind, ens, autres in (("FP", predictions - references, references), ("FN", references - predictions, predictions)):
        for x in sorted(ens):
            i, s, e, lab = x
            voisins = [y for y in autres if chevauche(x, y)]
            if not voisins:
                fam, face = ("faux positif" if kind == "FP" else "faux négatif"), ""
            else:
                meme_span = [y for y in voisins if y[1:3] == (s, e)]
                meme_lab = [y for y in voisins if y[3] == lab]
                fam = ("erreur de classification" if meme_span
                       else "erreur de frontière" if meme_lab else "erreur de frontière et d'étiquette")
                face = " ; ".join(f"{texte[i][y[1]:y[2]]} [{y[3]}]" for y in voisins)
            lignes.append({"type": kind, "id": i, "label": lab, "attendu_ou_predit": texte[i][s:e], "en_face": face,
                           "phrase": texte[i], "famille": fam,
                           "impact": "précision" if fam == "faux positif" else "rappel" if fam == "faux négatif" else "précision et rappel"})
    return pd.DataFrame(lignes)


tout = famille_erreurs(meilleur, REFS_TEST)
print(tout.groupby(["famille", "type"]).size().to_string())
cand = tout.sort_values(["famille", "label", "id"]).reset_index(drop=True)
cand["rang"] = cand.groupby(["famille", "label"]).cumcount()
choix = cand.sort_values(["rang", "famille", "label"]).head(14).drop(columns="rang")

# 2) À compléter À LA MAIN : cause probable et amélioration proposée (type_erreur est prérempli avec la famille ; affinez-le si besoin)
erreurs = choix.assign(type_erreur=choix["famille"], cause_probable="", amelioration_proposee="")[
    ["type", "id", "label", "attendu_ou_predit", "en_face", "phrase", "type_erreur", "impact", "cause_probable", "amelioration_proposee"]]
display(erreurs)

# 3) Sauvegarde (attention : relancer cette cellule écrase vos colonnes remplies ; travaillez ensuite dans le CSV)
erreurs.to_csv(RESULTS / "erreurs_analysees.csv", index=False, encoding="utf-8-sig")


## 11 Étape 9 – Démonstration (extension facultative)

Interface minimale : on colle un texte, on obtient les entités surlignées (`displacy`). Durée de la
démonstration : 2 à 4 minutes, 3 minutes en soutenance.

In [ ]:
from spacy import displacy

SYSTEMES = {"A – spaCy seul": nlp, "B – + règles génériques": nlp_b,
            "C – + lexique local": nlp_c, "D – spaCy ré-entraîné": nlp_d}


def demo(texte, systemes=None):
    """Affiche les entités d'un texte avec un ou plusieurs systèmes (par défaut : A, B, C et D)."""
    if isinstance(systemes, dict):
        choisis = systemes
    elif systemes is None:
        choisis = SYSTEMES
    else:                                    # un seul modèle spaCy passé directement
        choisis = {"système choisi": systemes}
    print("Texte :", texte)
    for nom, modele in choisis.items():
        print("\n" + nom)
        displacy.render(modele(texte), style="ent", jupyter=True)


if nlp is not None:
    demo("L'Université de Djibouti propose le Master IAMD à Djibouti depuis le 24 novembre 2026.")
    demo("L'Université de Djibouti a signé une convention avec l'Agence Française de Développement le 24 novembre 2026 à Djibouti.", nlp_c)


## 12 Étape 10 – Tests automatiques et sauvegardes

Au moins quatre tests (cf. TP2). Ceux du corpus sont dans `tests/test_corpus.py` ; ajoutez-en sur vos
règles (ex. « la baseline ne contient aucun nom propre du corpus »).

In [ ]:
# --- Test 1 : la baseline B ne contient aucun nom propre du corpus (seuls les mots-têtes génériques sont permis)
GENERIQUES = {"master", "licence", "dut", "bts", "deug", "doctorat"}
def litteraux(patterns):
    out = set()
    for p in patterns:
        for tok in p["pattern"]:
            for v in tok.values():
                if isinstance(v, str) and not v.startswith("^"):
                    out.add(v.lower())
                elif isinstance(v, dict) and "IN" in v:
                    out |= {x.lower() for x in v["IN"]}
    return out

noms_corpus = {e["text"].lower() for d in train + val + test for e in d["entities"]}
fuite = (litteraux(patterns_B) & noms_corpus) - GENERIQUES
assert not fuite, f"noms du corpus dans les règles de B : {fuite}"
print("OK : aucune forme d'entité du corpus dans les règles génériques de B")

# --- Figures (sauvegardées dans figures/)
import matplotlib.pyplot as plt
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

if resultats:
    # F1 par étiquette, par système
    f1 = pd.DataFrame({k: r["f1"] for k, r in resultats.items()}).loc[LABELS + ["micro", "macro"]]
    ax = f1.plot.bar(figsize=(8, 4), rot=0)
    ax.set_ylabel("F1 (correspondance exacte)"); ax.set_ylim(0, 1); ax.set_title("F1 par étiquette et système")
    plt.tight_layout(); plt.savefig(FIG / "f1_par_etiquette.png", dpi=150); plt.show()

    # précision / rappel / F1 micro
    micro = pd.DataFrame({k: r.loc["micro", ["precision", "rappel", "f1"]] for k, r in resultats.items()}).T
    ax = micro.plot.bar(figsize=(6, 4), rot=0)
    ax.set_ylim(0, 1); ax.set_title("Scores micro par système")
    plt.tight_layout(); plt.savefig(FIG / "scores_micro.png", dpi=150); plt.show()

## 13 Synthèse expérimentale

| Système | Ce qui fonctionne | Erreur observée | Prochaine expérience |
|---|---|---|---|
| A – spaCy seul | | | |
| B – + règles génériques | | | |
| C – + lexique local | | | |
| D – spaCy ré-entraîné | | | |

## 14 Conclusion (8 à 12 lignes)

*À rédiger : réponse à la question de recherche, limites (petite taille, peu de sources, test hétérogène, annotations
pré-remplies puis relues, FORM peu représentée (19 entités), déséquilibre des classes), perspectives.*

## 15 Livrables à déposer le lundi 23 novembre avant 18 h

- [ ] rapport de 6 à 8 pages (`rapport/`)
- [ ] ce notebook exécuté de bout en bout
- [ ] `README.md` et `requirements.txt`
- [ ] échantillon des données + `DATA_CARD.md` + `data/raw/SOURCES.md`
- [ ] `results/tableau_comparatif.csv`
- [ ] `results/erreurs_analysees.csv` (≥ 10 erreurs)
- [ ] démonstration de 2 à 4 minutes
- [ ] 8 à 10 diapositives (`slides/`)

## 16 Barème du jury (20 points)

| Critère | Points |
|---|---|
| Problématique et cadrage | 2 |
| Données et qualité | 3 |
| Baseline et méthodologie | 3 |
| Approche améliorée | 3 |
| Évaluation quantitative | 3 |
| Analyse d'erreurs et limites | 2 |
| Reproductibilité et code | 2 |
| Présentation et démonstration | 2 |
| **Total** | **20** |

## 17 Questions de synthèse à l'oral
1. Pourquoi sépare-t-on dev et test, et comment l'avez-vous fait ?
2. Pourquoi un lexique local peut-il améliorer le rappel tout en dégradant la précision ?
3. Pourquoi comparer à une baseline plutôt que de ne présenter que le meilleur système ?
4. Quelles erreurs persistent et que révèlent-elles sur le modèle ?
5. Que changeriez-vous avec plus de temps et plus de données ?